# Oil Spill Detection from SAR
**PyGeoVision v2.0 — Notebook 14**

Detect oil spills day/night through clouds using Sentinel-1 SAR adaptive threshold detection.

```bash
pip install "pygeovision[geo]"
```

In [ ]:
import pygeovision as pgv
import numpy as np, matplotlib.pyplot as plt
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE_DIR = Path("./outputs/14_oil_spill"); BASE_DIR.mkdir(parents=True,exist_ok=True)
BBOX = (-90.5, 28.0, -89.5, 29.0)
client = pgv.PyGeoVision()
print(client); print("\nStudy: Gulf of Mexico | Sensor: Sentinel-1 SAR (C-band VV+VH)")
print("Physics: oil dampens capillary waves -> lower SAR backscatter")

## Step 1: SAR Data Acquisition

In [ ]:
sar = client.search(bbox=BBOX, date_range=("2024-01-01","2024-12-31"),
    providers=["planetary_computer"], collections=["sentinel-1-rtc"],
    cloud_cover_max=100, limit=10)
print(f"SAR scenes: {len(sar)}")
for r in sar[:4]:
    print(f"  {r.date}  {r.satellite}")
sar_scene = None
if sar:
    dl = client.download(sar[:1],output_dir=BASE_DIR/"sar",
                          post_process=["reproject:EPSG:32615","cog"])
    if dl and dl[0].success: sar_scene = dl[0].path

## Step 2: Synthetic SAR + Adaptive Threshold

In [ ]:
np.random.seed(99); H=W=512
sar_vv = np.random.normal(-15,3.2,(H,W))
oil_mask = np.zeros((H,W),dtype=bool)
for cy,cx,ry,rx in [(90,90,45,38),(140,180,32,25),(195,95,28,22)]:
    y,x = np.ogrid[:H,:W]
    oil_mask[((y-cy)/ry)**2+((x-cx)/rx)**2 < 1.0] = True
rng = np.random.default_rng(99)
sar_vv[oil_mask] -= rng.uniform(8,15,oil_mask.sum())

bg = sar_vv[~oil_mask]
threshold = bg.mean() - 2.5*bg.std()
detected  = (sar_vv < threshold).astype(np.uint8)
PIXEL_M   = 10
spill_km2 = detected.sum()*PIXEL_M**2/1e6
est_tonnes = spill_km2*1e6*0.00008*870/1000

print(f"Background: {bg.mean():.1f} dB (mean)  {bg.std():.1f} dB (std)")
print(f"Threshold : {threshold:.1f} dB  (mean - 2.5*std)")
print(f"Spill area: {spill_km2:.2f} km2")
print(f"Est. volume: {est_tonnes:.0f} tonnes crude")
print(f"\nGeoAI API: client.geoai.segment.oil_spill(sar_scene)")

## Step 3: Visualisation

In [ ]:
fig,axes = plt.subplots(1,3,figsize=(15,5))
im1=axes[0].imshow(sar_vv,cmap='gray',vmin=-30,vmax=-5)
axes[0].set_title("SAR VV\n(dark=low backscatter=oil)",fontsize=11,fontweight='bold')
axes[0].axis('off'); plt.colorbar(im1,ax=axes[0],fraction=0.046,pad=0.04,label='dB')
axes[1].imshow(detected,cmap='Reds')
axes[1].set_title(f"Oil Spill Detection\n({spill_km2:.1f} km2 flagged)",fontsize=11,fontweight='bold'); axes[1].axis('off')
comp = np.zeros((*sar_vv.shape,3))
norm = (sar_vv-sar_vv.min())/(sar_vv.max()-sar_vv.min())
comp[:,:,0]=comp[:,:,1]=comp[:,:,2]=norm
comp[detected>0]=[1,0.1,0.1]
axes[2].imshow(comp); axes[2].axis('off')
axes[2].set_title("SAR + Oil Overlay\nRed = detected",fontsize=11,fontweight='bold')
bg_vals = sar_vv[~oil_mask].ravel()
plt.suptitle(f"Oil Spill Detection — Gulf of Mexico (Sentinel-1 SAR)\n{spill_km2:.1f} km2  ~{est_tonnes:.0f} tonnes",fontsize=12,fontweight='bold')
plt.tight_layout(); plt.savefig(BASE_DIR/"oil_spill.png",dpi=150,bbox_inches='tight'); plt.show()

## Summary

In [ ]:
print("="*60); print("NOTEBOOK 14 — Oil Spill Detection (SAR)"); print("="*60)
print(f"Sensor: Sentinel-1 SAR | Threshold: {threshold:.1f} dB")
print(f"Spill : {spill_km2:.2f} km2  ~{est_tonnes:.0f} tonnes")
print("\nAdvantages over optical: night + cloud penetration")
print("Next: 15_air_quality_monitoring.ipynb")